# Reconnaissance et analyse automatique d'expressions faciales

**Fondamentaux du Deep Learning** - Master, Semestre 1 (2026-2027)

**Binome :** Mon Watts ([@damonwatts11](https://github.com/damonwatts11)) - Huet ([@SpectreAH](https://github.com/SpectreAH))  
**Formatrice :** Hanane Zerdoum - **Soutenance :** vendredi 9 octobre 2026

Dataset retenu : **FER+ (FERPlus)** - images FER-2013 avec labels re-annotes (8 classes).

---

## Plan du notebook

| Partie | Contenu | Statut |
|:------:|---------|--------|
| 0 | Configuration et imports | - |
| 1 | Recherche, comprehension et preparation des donnees | Obligatoire |
| 2 | Modele de reference (reseau dense) | Obligatoire |
| 3 | Construction d'un CNN avec Keras | Obligatoire |
| 4 | Entrainement du reseau | Obligatoire |
| 5 | Evaluation et analyse des erreurs | Obligatoire |
| 6 | Experimentation et amelioration (>= 3 experiences) | Obligatoire |
| 7 | Enrichissement avec les nouvelles notions | Enrichissement |
| 8 | Detection de plusieurs visages (YOLO) | Extension |
| 9 | Extension a la video | Extension (bonus) |


## 0 - Configuration et imports

Imports communs, graine aleatoire (reproductibilite) et constantes globales du projet.
Sur Google Colab, activez le GPU : *Execution -> Modifier le type d'execution -> GPU*.

In [ ]:
# Decommenter sur Colab si besoin :
# !pip install -q ultralytics seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow', tf.__version__)
print('GPU disponible :', tf.config.list_physical_devices('GPU'))

In [ ]:
# --- Reproductibilite ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Constantes du projet ---
IMG_SIZE    = (48, 48)   # FER+ : images 48x48
CHANNELS    = 1          # niveaux de gris
BATCH_SIZE  = 64
EPOCHS      = 30

# Classes FER+ (8). Equivalents FR : neutre, joie, surprise, tristesse,
# colere, degout, peur, mepris.
CLASS_NAMES = ['neutral', 'happiness', 'surprise', 'sadness',
               'anger', 'disgust', 'fear', 'contempt']
NUM_CLASSES = len(CLASS_NAMES)

## 1 - Recherche, comprehension et preparation des donnees

**Dataset : FER+ (FERPlus)** - re-annotation par Microsoft du jeu FER-2013.
Chaque image 48x48 en niveaux de gris a recu **10 votes** d'annotateurs, ce qui
donne des labels plus fiables que FER-2013 d'origine.

- Source : [github.com/microsoft/FERPlus](https://github.com/microsoft/FERPlus) (labels) + FER-2013 sur Kaggle (pixels). Licence : voir le repo.
- 8 classes : neutral, happiness, surprise, sadness, anger, disgust, fear, contempt.
- ~35 000 images, fort **desequilibre** (happiness/neutral majoritaires, disgust/contempt rares) -> important pour la Partie 5.

### Deux fichiers a recuperer, a placer dans `data/`
| Fichier | Contenu | Source |
|---------|---------|--------|
| `fer2013.csv` | pixels (`emotion, pixels, Usage`) | Kaggle *FER-2013* (compte requis) |
| `fer2013new.csv` | votes FER+ (`Usage, Image name, neutral, ..., NF`) | repo microsoft/FERPlus |

Les deux fichiers ont **le meme nombre de lignes, dans le meme ordre** : on les
aligne par index (la ligne i de l'un correspond a la ligne i de l'autre).

In [ ]:
# --- Recuperation des fichiers ---
# fer2013new.csv (labels FER+, public) :
# !wget -q https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv -O data/fer2013new.csv
#
# fer2013.csv (pixels) : telecharger depuis Kaggle et placer dans data/
# (Kaggle 'FER-2013' ou 'Challenges in Representation Learning: FER-2013').

DATA_DIR = 'data'
fer     = pd.read_csv(os.path.join(DATA_DIR, 'fer2013.csv'))      # emotion, pixels, Usage
ferplus = pd.read_csv(os.path.join(DATA_DIR, 'fer2013new.csv'))   # Usage, Image name, votes...

assert len(fer) == len(ferplus), 'Les deux fichiers doivent avoir le meme nombre de lignes'
print('Lignes :', len(fer))
ferplus.head()

In [ ]:
# --- Construction des labels (vote majoritaire) ---
votes = ferplus[CLASS_NAMES].to_numpy()                 # (N, 8) votes par emotion
unknown_nf = ferplus[['unknown', 'NF']].to_numpy().sum(axis=1)

# On garde les images dont une vraie emotion l'emporte sur unknown/NF.
keep = (votes.sum(axis=1) > 0) & (votes.max(axis=1) >= unknown_nf)
y_all = votes.argmax(axis=1)                            # classe = emotion majoritaire

# --- Pixels -> images 48x48x1 ---
def parse_pixels(s):
    return np.asarray(s.split(), dtype='float32').reshape(48, 48, 1)

X_all = np.stack(fer['pixels'].map(parse_pixels).to_numpy())
usage = fer['Usage'].to_numpy()
print('Images gardees :', int(keep.sum()), '/', len(keep))

In [ ]:
# --- Split officiel FER-2013 via la colonne Usage ---
def subset(split_name):
    m = keep & (usage == split_name)
    return X_all[m], y_all[m]

X_train, y_train = subset('Training')     # entrainement
X_val,   y_val   = subset('PublicTest')   # validation
X_test,  y_test  = subset('PrivateTest')  # test (jamais utilise pour valider)

print('train/val/test :', X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# --- Exploration : distribution des classes (train) ---
counts = pd.Series(y_train).map(dict(enumerate(CLASS_NAMES))).value_counts()
counts = counts.reindex(CLASS_NAMES)
counts.plot(kind='bar'); plt.title('Images par classe (train)'); plt.ylabel('nb'); plt.show()
counts

In [ ]:
# --- Affichage d'exemples par categorie ---
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(10, 2 * NUM_CLASSES))
for i, cls in enumerate(CLASS_NAMES):
    idx = np.where(y_train == i)[0][:5]
    for j, k in enumerate(idx):
        axes[i, j].imshow(X_train[k].squeeze(), cmap='gray'); axes[i, j].axis('off')
    axes[i, 0].set_ylabel(cls, rotation=0, ha='right', va='center')
plt.tight_layout(); plt.show()

### Preparation des donnees

Role de chaque etape : **normalisation** des pixels ([0,1]), **encodage** one-hot des
labels pour la classification multiclasse. Les images sont deja en 48x48 (pas de
redimensionnement). Le decoupage train/validation/test vient du champ `Usage`.

> Le jeu de **test** est reserve a l'evaluation finale : il ne sert jamais de validation.

In [ ]:
# --- Normalisation + one-hot ---
X_train = X_train / 255.0
X_val   = X_val   / 255.0
X_test  = X_test  / 255.0

y_train = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

print('X_train', X_train.shape, 'y_train', y_train.shape)

## 2 - Modele de reference (reseau dense)

Reseau simple servant de **point de comparaison** avec le CNN.  
`Image -> Flatten -> Dense + activation -> Dense -> Softmax`

A savoir expliquer : transformation de l'image en entree, poids et biais, propagation avant, fonctions d'activation, fonction de perte, retropropagation, descente de gradient, sortie multiclasse (Softmax).

In [ ]:
def build_baseline():
    model = keras.Sequential([
        keras.Input(shape=(*IMG_SIZE, CHANNELS)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='baseline_dense')
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

baseline = build_baseline()
baseline.summary()

## 3 - Construction d'un CNN avec Keras

Proposez **et justifiez** votre propre architecture (plusieurs blocs conv + pooling).  
`Conv2D+ReLU -> MaxPooling -> Conv2D+ReLU -> MaxPooling -> Flatten -> Dense -> Softmax`

A expliquer : filtres, convolution, feature maps, kernel, stride, padding, ReLU, pooling, Flatten, Dense, couche de sortie. Suivez l'evolution des **dimensions** et le **nombre de parametres** couche par couche (`model.summary()`).

In [ ]:
def build_cnn():
    model = keras.Sequential([
        keras.Input(shape=(*IMG_SIZE, CHANNELS)),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='cnn')
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn = build_cnn()
cnn.summary()

## 4 - Entrainement du reseau

Justifiez : fonction de perte, optimiseur, batch size, nombre d'epoques, metriques.  
Etudiez les courbes **loss / accuracy** (train vs validation) et identifiez un eventuel **surapprentissage**.

> Classes desequilibrees : on peut passer `class_weight` a `fit()` (voir plus bas).

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5),
]

# Optionnel : poids de classe pour contrer le desequilibre
# from sklearn.utils.class_weight import compute_class_weight
# yt = y_train.argmax(1)
# cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=yt)
# class_weight = dict(enumerate(cw))

history = cnn.fit(X_train, y_train,
                  validation_data=(X_val, y_val),
                  epochs=EPOCHS, batch_size=BATCH_SIZE,
                  callbacks=callbacks)  # , class_weight=class_weight

In [ ]:
def plot_history(history):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h['loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
    ax[0].set_title('Loss'); ax[0].legend()
    ax[1].plot(h['accuracy'], label='train'); ax[1].plot(h['val_accuracy'], label='val')
    ax[1].set_title('Accuracy'); ax[1].legend()
    plt.show()

plot_history(history)

## 5 - Evaluation et analyse des erreurs

L'accuracy seule ne suffit pas (classes desequilibrees). Produisez une **matrice de confusion** et analysez : expressions bien reconnues, souvent confondues, les plus difficiles. Affichez des exemples `image + vraie classe + classe predite + probabilite`, dont plusieurs **erreurs**.

In [ ]:
# --- Evaluation sur le test ---
test_loss, test_acc = cnn.evaluate(X_test, y_test)
print('Test accuracy :', round(test_acc, 4))

y_pred = cnn.predict(X_test)
y_pred_cls = y_pred.argmax(1)
y_true_cls = y_test.argmax(1)

print(classification_report(y_true_cls, y_pred_cls, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true_cls, y_pred_cls)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predit'); plt.ylabel('Vrai'); plt.show()

In [ ]:
# --- Exemples de predictions (dont erreurs) ---
idx = np.random.choice(len(X_test), 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, k in zip(axes.ravel(), idx):
    ax.imshow(X_test[k].squeeze(), cmap='gray'); ax.axis('off')
    vrai, pred = CLASS_NAMES[y_true_cls[k]], CLASS_NAMES[y_pred_cls[k]]
    p = y_pred[k].max()
    couleur = 'green' if vrai == pred else 'red'
    ax.set_title(f'{vrai}\n-> {pred} ({p:.2f})', color=couleur, fontsize=9)
plt.tight_layout(); plt.show()

## 6 - Experimentation et amelioration

**Au minimum 3 experiences**, en ne modifiant qu'un nombre limite de parametres a la fois (couches, filtres, kernels, pooling, neurones, batch size, learning rate, optimiseur, epoques) ou techniques (Dropout, Data Augmentation, Early Stopping, regularisation).

Presentez un **tableau comparatif** et justifiez le modele final.

| # | Modification | Val. accuracy | Observations |
|:-:|--------------|:-------------:|--------------|
| 1 | _(baseline CNN)_ | | |
| 2 | | | |
| 3 | | | |

In [ ]:
# --- Exemple : Data Augmentation ---
# data_aug = keras.Sequential([
#     layers.RandomFlip('horizontal'),
#     layers.RandomRotation(0.1),
#     layers.RandomZoom(0.1),
# ])
# Inserer data_aug juste apres la couche Input du CNN pour l'experience correspondante.

## 7 - Enrichissement avec les nouvelles notions

Pistes : Data Augmentation, **Transfer Learning** (reseaux pre-entraines), regularisation, detection d'objets, YOLO.

In [ ]:
# --- Exemple : Transfer Learning (necessite CHANNELS = 3) ---
# base = keras.applications.MobileNetV2(
#     input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
# base.trainable = False
# tl = keras.Sequential([
#     base, layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation='relu'), layers.Dropout(0.3),
#     layers.Dense(NUM_CLASSES, activation='softmax')])
# tl.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## 8 - Extension : detection de plusieurs visages (YOLO)

`Image -> detection des visages -> bounding boxes -> extraction -> CNN -> expressions`

**Attention** : un YOLO standard (COCO) detecte des *personnes*, pas des *visages*. Utilisez un modele entraine aux visages (ou justifiez une autre solution). Chaque visage extrait doit subir **exactement le meme pretraitement** que les images d'entrainement (48x48, gris, /255).

Notions : classification vs detection, bounding box, confidence score, IoU, NMS, modele pre-entraine, (fine-tuning), precision/recall/mAP.

In [ ]:
# from ultralytics import YOLO
# detector = YOLO('yolov8n-face.pt')  # modele de detection de visages
#
# def analyser_image(img):
#     results = detector(img)
#     for box in results[0].boxes:
#         x1, y1, x2, y2 = map(int, box.xyxy[0])
#         visage = img[y1:y2, x1:x2]
#         # MEME pretraitement que l'entrainement (resize 48x48, gris, /255)
#         # pred = cnn.predict(prep(visage))
#         # dessiner la box + CLASS_NAMES[pred.argmax()] + score
#     return img

## 9 - Extension a la video (bonus)

`Video -> images successives -> detection -> extraction -> CNN -> predictions -> affichage`

In [ ]:
# import cv2
# cap = cv2.VideoCapture('data/demo.mp4')
# while cap.isOpened():
#     ok, frame = cap.read()
#     if not ok: break
#     frame = analyser_image(frame)
#     # ecrire / afficher la frame annotee
# cap.release()

## Conclusion

Neurone -> MLP -> Backpropagation -> Keras -> CNN -> Convolution -> Classification multiclasse -> Softmax -> Amelioration -> Detection -> YOLO -> Application complete de vision par ordinateur.

**Livrables (soutenance 9 oct. 2026)** : notebook executable de bout en bout, presentation, demonstration du modele final.